# Lectures 18 and 19 — Road Crashes, Spatial Joins, Layer Order and Folium

In these lectures we continue working with **spatial joins** using road-crash data from Washington, D.C.

The main ideas are:

- assign the same crash points to different spatial units;
- understand what the original fields in each spatial layer mean;
- aggregate and map crashes at ward and census-tract levels;
- normalize crash counts by polygon area;
- practise the same workflow using neighborhood clusters;
- understand how `zorder` controls the drawing order of layers;
- move from static GeoPandas maps to interactive Folium maps;
- use an interactive map to inspect crash points that did not match a census tract.

## Set up the notebook

All files used here are stored in:

`C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data\DC`

If your files are stored somewhere else, change `data_path` to the folder on your own computer.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data\DC"

# Revision

We will use the 2024–2025 road-crash file together with three polygon layers:

- **Wards**
- **Census tracts**
- **Neighborhood clusters**

The crash file contains point observations. The other files divide Washington, D.C. into different spatial units.

The same crash can therefore be described at different spatial scales. A crash may belong to a particular ward, a particular census tract, and a particular neighborhood cluster.

For these examples, the crash file already contains valid point geometry, so we will use its existing `geometry` column directly.

In [ ]:
crashes = gpd.read_file(data_path + r"\Crashes_2024_2025.geojson")
wards = gpd.read_file(data_path + r"\Wards_from_2022.geojson")
tracts = gpd.read_file(data_path + r"\Census_Tracts_in_2020.geojson")
clusters = gpd.read_file(data_path + r"\Neighborhood_Clusters.geojson")

In [ ]:
print("Crashes:", crashes.shape)
print("Wards:", wards.shape)
print("Census tracts:", tracts.shape)
print("Neighborhood clusters:", clusters.shape)

## Understanding the original field names

We will **keep the original column names** in the spatial files. Real GIS datasets often contain several identifiers, and it is useful to understand what they represent instead of immediately renaming them.

### Wards

Some useful fields are:

- `WARD` — the ward number;
- `NAME` — the readable ward name, such as *Ward 1*;
- `WARD_ID` — another ward identifier stored in the source GIS layer;
- `OBJECTID` — the internal record identifier used by the GIS dataset;
- `geometry` — the ward polygon.

### Census tracts

Some useful fields are:

- `TRACT` — the census-tract code;
- `GEOID` — the full geographic identifier for the census tract;
- `NAME` — a readable tract name;
- `OBJECTID` — the internal record identifier;
- `geometry` — the census-tract polygon.

### Neighborhood clusters

Some useful fields are:

- `NAME` — the cluster name, such as *Cluster 16*;
- `NBH_NAMES` — the neighborhood names grouped within that cluster;
- `GIS_ID` — an identifier from the GIS layer;
- `OBJECTID` — the internal record identifier;
- `geometry` — the cluster polygon.

The important point is that we do not have to rename every field. We first understand what the source fields represent, and then select only the fields needed for a particular analysis.

In [ ]:
display(wards[["WARD", "NAME", "WARD_ID", "OBJECTID"]].head())
display(tracts[["TRACT", "GEOID", "NAME", "OBJECTID"]].head())
display(clusters[["NAME", "NBH_NAMES", "GIS_ID", "OBJECTID"]].head())

## A smaller set of crash columns for the spatial joins

The crash file has many attributes. For the spatial-join examples we only need a manageable set of them.

Notice that the original crash file itself contains a `WARD` field. We leave that field out of `crash_base` because the purpose of the next example is to **derive the ward from the crash geometry**.

If both the left and right datasets contain a field with the same name, GeoPandas may create suffixes such as `_left` and `_right`. Selecting only the fields needed for the task keeps the output easier to read.

In [ ]:
crash_base = crashes[
    [
        "CRIMEID",
        "REPORTDATE",
        "ADDRESS",
        "LATITUDE",
        "LONGITUDE",
        "TOTAL_VEHICLES",
        "TOTAL_PEDESTRIANS",
        "TOTAL_BICYCLES",
        "SPEEDING_INVOLVED",
        "MAJORINJURIES_DRIVER",
        "MINORINJURIES_DRIVER",
        "MAJORINJURIES_PEDESTRIAN",
        "MINORINJURIES_PEDESTRIAN",
        "FATAL_DRIVER",
        "FATAL_PEDESTRIAN",
        "geometry"
    ]
].copy()

crash_base.head()

# Spatial join with wards

The question is:

> **For each crash point, which ward polygon contains it?**

The crash points are on the left. The ward polygons are on the right. Therefore `predicate="within"` reads naturally as:

> Is this crash point **within** this ward polygon?

We keep the original ward fields `WARD` and `NAME`.

In [ ]:
crash_ward = gpd.sjoin(
    crash_base,
    wards[["WARD", "NAME", "geometry"]],
    how="left",
    predicate="within"
)

crash_ward[
    ["CRIMEID", "ADDRESS", "WARD", "NAME"]
].head()

In [ ]:
print("Rows after the spatial join:", len(crash_ward))
print("Crashes without a ward match:", crash_ward["WARD"].isna().sum())

## Count crashes in each ward

After the spatial join, every matched crash carries the ward number and ward name.

We can now group the crash records by `WARD` and count how many crash records fall in each ward.

In [ ]:
ward_counts = (
    crash_ward
    .dropna(subset=["WARD"])
    .groupby("WARD")
    .size()
    .reset_index(name="crash_count")
)

ward_counts

The count table is then merged back to the ward polygons.

This is now a normal **attribute merge** because both tables share the field `WARD`.

In [ ]:
ward_map = wards.merge(
    ward_counts,
    on="WARD",
    how="left"
)

ward_map["crash_count"] = ward_map["crash_count"].fillna(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

ward_map.plot(
    ax=ax,
    column="crash_count",
    legend=True,
    edgecolor="white"
)

ax.set_title("Road Crashes by Ward, 2024–2025")
ax.set_axis_off()

plt.show()

# Spatial join with census tracts

Now we ask a more detailed spatial question:

> **Which census tract contains each crash?**

Census tracts are much smaller than wards, so the resulting map gives a finer spatial representation of crash locations.

The spatial-join logic is exactly the same. Only the polygon layer on the right changes.

In [ ]:
crash_tract = gpd.sjoin(
    crash_base,
    tracts[["TRACT", "GEOID", "NAME", "geometry"]],
    how="left",
    predicate="within"
)

crash_tract[
    ["CRIMEID", "ADDRESS", "TRACT", "GEOID", "NAME"]
].head()

In [ ]:
print("Rows after the spatial join:", len(crash_tract))
print("Crashes without a census-tract match:", crash_tract["GEOID"].isna().sum())

Some crash points do not match a census-tract polygon.

We will **not immediately delete these records**. Later, Folium will let us inspect where these unmatched points are located.

First, we aggregate the matched crashes by census tract.

In [ ]:
tract_counts = (
    crash_tract
    .dropna(subset=["GEOID"])
    .groupby("GEOID")
    .size()
    .reset_index(name="crash_count")
)

tract_map = tracts.merge(
    tract_counts,
    on="GEOID",
    how="left"
)

tract_map["crash_count"] = tract_map["crash_count"].fillna(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

tract_map.plot(
    ax=ax,
    column="crash_count",
    legend=True,
    edgecolor="white",
    linewidth=0.4
)

ax.set_title("Road Crashes by Census Tract, 2024–2025")
ax.set_axis_off()

plt.show()

## Normalizing the crash count by area

A raw crash count answers:

> **How many crashes occurred in this tract?**

But census tracts have different physical sizes. A large tract has more land area than a small tract.

One simple normalization is:

> **Crash density = number of crashes / area of the tract**

This gives the number of crashes per square kilometre.

This is useful for comparing the **spatial concentration** of crashes. It should not be interpreted as a complete measure of crash risk. A true risk measure might also consider traffic volume, road length, population, or some other measure of exposure.

We must calculate area in a **projected CRS**, because latitude and longitude are measured in degrees. Washington, D.C. lies in UTM Zone 18N, so we use `EPSG:32618`.

Dividing by `1_000_000` converts square metres to square kilometres:

`1 km² = 1,000,000 m²`

In [ ]:
tract_area = tract_map.to_crs(epsg=32618)

tract_area["area_km2"] = (
    tract_area.geometry.area / 1_000_000
)

tract_area["crashes_per_km2"] = (
    tract_area["crash_count"] /
    tract_area["area_km2"]
)

tract_area[
    ["TRACT", "NAME", "crash_count", "area_km2", "crashes_per_km2"]
].head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

tract_area.plot(
    ax=ax,
    column="crashes_per_km2",
    legend=True,
    edgecolor="white",
    linewidth=0.4
)

ax.set_title("Road Crash Density by Census Tract, 2024–2025")
ax.set_axis_off()

plt.show()

Compare the two census-tract maps:

- `crash_count` shows the **number** of crashes;
- `crashes_per_km2` shows the **number of crashes relative to tract area**.

A tract can have a moderate crash count but a high crash density if it covers a very small area.

This is an important reminder that the way we summarize spatial data can change the pattern we see.

# Class Exercise 15 — Spatial join with neighborhood clusters

Now repeat the same workflow using `Neighborhood_Clusters.geojson`.

The useful fields are:

- `NAME` — the cluster name;
- `NBH_NAMES` — the neighborhoods contained in that cluster;
- `geometry` — the cluster polygon.

### Tasks

1. Spatially join the crash points to the neighborhood clusters using `predicate="within"`.
2. Display `CRIMEID`, `ADDRESS`, `NAME`, and `NBH_NAMES` for the first few rows.
3. Find how many crash records did not match a neighborhood cluster.
4. Count the number of crashes in each neighborhood cluster.
5. Merge the counts back to the neighborhood-cluster polygons.
6. Create a choropleth showing the **number of crashes** by neighborhood cluster.
7. Reproject the neighborhood-cluster map to `EPSG:32618`.
8. Calculate the area of each cluster in square kilometres.
9. Calculate **crashes per km²**.
10. Create a second choropleth showing the area-normalized crash density.

### Think about

Compare the two maps.

- Does the cluster with the highest number of crashes also have the highest crashes per km²?
- Why can the pattern change after normalizing by area?

## Same crashes, different spatial summaries

We have now used the same crash points with three different polygon layers:

- **Wards** give a broad summary;
- **Census tracts** provide a much finer spatial summary;
- **Neighborhood clusters** provide another planning-oriented geography.

The crash locations themselves have not changed. What changes is the spatial unit used to summarize them.

This is why the **spatial unit of analysis matters**. The pattern we see can change depending on the polygons used for aggregation.

# Understanding `zorder`

So far, we have mainly drawn one spatial layer at a time. Real maps often contain several layers together:

- polygon fills;
- administrative boundaries;
- point observations;
- labels or annotations.

When layers overlap, we need to decide **which one appears on top**.

Matplotlib uses `zorder` to control this drawing order.

> **Higher `zorder` = drawn above layers with lower `zorder`.**

For example:

- `zorder=1` can be a background polygon;
- `zorder=2` can be another set of boundaries;
- `zorder=3` can be point observations.

The numbers themselves have no units.

`zorder=3` is not three times more important than `zorder=1`.

Only the **relative order** matters.

Therefore, these two sets produce the same stacking order:

`1, 2, 3`

and

`10, 20, 30`

## A map with three layers

For visualisation only, we use a sample of crash points so that the map is not overwhelmed by tens of thousands of points.

The earlier spatial joins used the full crash dataset.

In [ ]:
crash_plot = crash_base.sample(
    3000,
    random_state=42
)

ward_z = 1
tract_z = 2
crash_z = 3

fig, ax = plt.subplots(figsize=(10, 10))

wards.plot(
    ax=ax,
    color="lightgray",
    edgecolor="black",
    linewidth=1.2,
    zorder=ward_z
)

tracts.boundary.plot(
    ax=ax,
    color="gray",
    linewidth=0.4,
    zorder=tract_z
)

crash_plot.plot(
    ax=ax,
    color="red",
    markersize=5,
    alpha=0.6,
    zorder=crash_z
)

ax.set_title("Layer Order Using zorder")
ax.set_axis_off()

plt.show()

In this map:

- the ward polygons have `zorder=1`, so they form the background;
- census-tract boundaries have `zorder=2`, so they are drawn above the wards;
- crash points have `zorder=3`, so they are drawn above both polygon layers.

Try changing the variables:

- set `crash_z = 0` — the crash points are placed underneath layers with higher `zorder`;
- set `ward_z = 5` — the ward polygons move above the tract boundaries and crash points if their values remain lower;
- set `tract_z = 10` — the census-tract boundaries are drawn above the other layers.

### `zorder` is different from `alpha`

`zorder` answers:

> **Which layer is above which layer?**

`alpha` answers:

> **How transparent is the layer?**

### `zorder` is also different from `markersize` and `linewidth`

- `markersize` controls the size of points;
- `linewidth` controls the thickness of lines;
- `zorder` controls the stacking order.

If two layers have the same `zorder`, plotting order can affect which one appears above the other. Giving different `zorder` values makes the intended layer order explicit.

# Lecture 19 — Introduction to Folium

GeoPandas and Matplotlib produce **static maps**. They are useful for analysis and publication-quality figures.

Folium creates **interactive web maps**.

An interactive Folium map allows the user to:

- zoom in and out;
- pan across the map;
- hover over features;
- click features to inspect information;
- turn data layers on and off.

Folium is a Python interface to the JavaScript mapping library **Leaflet**. We write Python code, and Folium generates the interactive web map.

## Creating the first Folium map

A Folium map needs a starting location and a starting zoom level.

Notice the coordinate order:

- GeoPandas/Shapely point creation commonly uses **longitude, latitude** or **x, y**;
- Folium `location=[...]` uses **latitude, longitude**.

For Washington, D.C., we can start with approximately:

`[38.9072, -77.0369]`

In [ ]:
m = folium.Map(
    location=[38.9072, -77.0369],
    zoom_start=11,
    control_scale=True
)

m

### Understanding the arguments in `folium.Map()`

`location=[38.9072, -77.0369]`

- sets the centre of the map;
- the order is `[latitude, longitude]`;
- these coordinates place the initial view around Washington, D.C.

`zoom_start=11`

- controls the initial zoom;
- a smaller value shows a larger geographic area;
- a larger value zooms closer to streets and individual locations;
- the user can still zoom in and out after the map opens.

`control_scale=True`

- adds a distance scale to the map.

We did **not** specify a tile layer, so Folium uses its default **OpenStreetMap** background.

## Background tiles

The background map in a web map is often called a **tile layer** or **basemap**.

OpenStreetMap is Folium's default and is sufficient for our examples.

Other tile services exist, for example light cartographic basemaps, terrain maps, and satellite imagery. Depending on the provider, some tile services may require:

- an API key;
- an access token;
- specific attribution;
- acceptance of the provider's terms of use.

Tile-service availability can also change over time.

For this lecture we will keep the default **OpenStreetMap** basemap and focus on adding our own spatial data.

## Add ward polygons

`folium.GeoJson()` can take a GeoDataFrame directly.

We define how the ward polygons should look using a small style function.

In [ ]:
def ward_style(feature):
    return {
        "fillColor": "lightgray",
        "color": "black",
        "weight": 1.5,
        "fillOpacity": 0.15
    }

The style function receives a GeoJSON feature and returns a dictionary describing how it should look.

- `fillColor` — colour inside the polygon;
- `color` — polygon boundary colour;
- `weight` — boundary thickness;
- `fillOpacity` — transparency of the polygon fill.

A low `fillOpacity` keeps the OpenStreetMap background visible underneath the ward polygons.

In [ ]:
folium.GeoJson(
    wards[["WARD", "NAME", "REP_NAME", "geometry"]],
    name="Wards",
    style_function=ward_style,
    tooltip=folium.GeoJsonTooltip(
        fields=["WARD", "NAME", "REP_NAME"],
        aliases=[
            "Ward number:",
            "Ward name:",
            "Representative:"
        ]
    )
).add_to(m)

## Add crash points

A static map can quickly become crowded when thousands of crashes are plotted.

For the interactive demonstration, we use a random sample of crash records. This is only for displaying the map; the earlier spatial joins used the full dataset.

We also keep several useful crash attributes so that they can appear when the user hovers over a point.

In [ ]:
crash_web = crashes.sample(
    1000,
    random_state=42
)[
    [
        "CRIMEID",
        "REPORTDATE",
        "ADDRESS",
        "TOTAL_VEHICLES",
        "TOTAL_PEDESTRIANS",
        "TOTAL_BICYCLES",
        "SPEEDING_INVOLVED",
        "MAJORINJURIES_DRIVER",
        "MINORINJURIES_DRIVER",
        "MAJORINJURIES_PEDESTRIAN",
        "MINORINJURIES_PEDESTRIAN",
        "FATAL_DRIVER",
        "FATAL_PEDESTRIAN",
        "geometry"
    ]
].copy()

# Folium/GeoJSON needs the date stored as text
crash_web["REPORTDATE"] = (
    crash_web["REPORTDATE"].astype(str)
)

crash_web["ADDRESS"] = (
    crash_web["ADDRESS"].fillna("Not recorded")
)

In [ ]:
folium.GeoJson(
    crash_web,
    name="Crash sample",
    marker=folium.CircleMarker(
        radius=3,
        color="darkred",
        weight=1,
        fill=True,
        fill_opacity=0.7
    ),
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "CRIMEID",
            "REPORTDATE",
            "ADDRESS",
            "TOTAL_VEHICLES",
            "TOTAL_PEDESTRIANS",
            "TOTAL_BICYCLES",
            "SPEEDING_INVOLVED",
            "MAJORINJURIES_DRIVER",
            "MINORINJURIES_DRIVER",
            "MAJORINJURIES_PEDESTRIAN",
            "MINORINJURIES_PEDESTRIAN",
            "FATAL_DRIVER",
            "FATAL_PEDESTRIAN"
        ],
        aliases=[
            "Crash ID:",
            "Report date:",
            "Address:",
            "Vehicles involved:",
            "Pedestrians involved:",
            "Bicycles involved:",
            "Speeding involved:",
            "Major driver injuries:",
            "Minor driver injuries:",
            "Major pedestrian injuries:",
            "Minor pedestrian injuries:",
            "Driver fatalities:",
            "Pedestrian fatalities:"
        ],
        sticky=False
    )
).add_to(m)

m

### Understanding the point-marker arguments

`marker=folium.CircleMarker(...)`

tells Folium how the point features should be drawn.

Inside `CircleMarker()`:

- `radius=3` — radius of each point in screen pixels;
- `color="darkred"` — colour of the point outline;
- `weight=1` — thickness of the outline;
- `fill=True` — fills the inside of the circle;
- `fill_opacity=0.7` — makes the fill partly transparent.

### Understanding `tooltip`

A **tooltip** appears when the user moves the mouse over a feature.

`fields=[...]`

- contains the actual column names from the GeoDataFrame;
- determines which attributes are shown.

`aliases=[...]`

- contains the labels shown to the user;
- the first alias corresponds to the first field, the second alias to the second field, and so on.

`sticky=False`

- means the tooltip behaves as a normal hover tooltip rather than remaining attached to the cursor.

### Why is there no `for` loop?

Many Folium examples use a loop because they add markers one row at a time.

Here, however, `crash_web` is already a GeoDataFrame containing all the point geometries and attributes we want to display.

`folium.GeoJson()` can take the **entire GeoDataFrame at once**.

Therefore, a row-by-row `for` loop is unnecessary for this example.

## Add a layer-control box

The map now contains:

- the OpenStreetMap basemap;
- ward polygons;
- crash points.

`LayerControl()` allows the user to turn the ward and crash layers on and off.

In [ ]:
folium.LayerControl().add_to(m)

m

# Using Folium to inspect unmatched census-tract crashes

Earlier, some crash points did not match a census tract using `predicate="within"`.

Instead of automatically deleting those records, we can inspect their locations.

This is one useful role of an interactive map: it can help us **diagnose spatial-data problems visually**.

In [ ]:
missing_tract = crash_tract[
    crash_tract["GEOID"].isna()
].copy()

print(
    "Number of unmatched census-tract crashes:",
    len(missing_tract)
)

missing_tract[
    ["CRIMEID", "ADDRESS", "LATITUDE", "LONGITUDE"]
].head()

We will create a second Folium map containing:

- all census-tract boundaries; and
- only the crash points that failed to match a census tract.

This lets us zoom in and see where those unmatched points actually lie.

Again, no loop is necessary because the unmatched crash records are already stored as a GeoDataFrame.

In [ ]:
missing_web = missing_tract[
    [
        "CRIMEID",
        "REPORTDATE",
        "ADDRESS",
        "LATITUDE",
        "LONGITUDE",
        "geometry"
    ]
].copy()

missing_web["REPORTDATE"] = (
    missing_web["REPORTDATE"].astype(str)
)

missing_web["ADDRESS"] = (
    missing_web["ADDRESS"].fillna("Not recorded")
)

In [ ]:
def tract_style(feature):
    return {
        "fillColor": "white",
        "color": "gray",
        "weight": 1,
        "fillOpacity": 0.05
    }

In [ ]:
m_missing = folium.Map(
    location=[38.9072, -77.0369],
    zoom_start=11,
    control_scale=True
)

folium.GeoJson(
    tracts[["TRACT", "GEOID", "NAME", "geometry"]],
    name="Census tracts",
    style_function=tract_style,
    tooltip=folium.GeoJsonTooltip(
        fields=["TRACT", "GEOID", "NAME"],
        aliases=[
            "Tract:",
            "GEOID:",
            "Name:"
        ]
    )
).add_to(m_missing)

folium.GeoJson(
    missing_web,
    name="Unmatched crash points",
    marker=folium.CircleMarker(
        radius=4,
        color="red",
        weight=1,
        fill=True,
        fill_opacity=0.8
    ),
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "CRIMEID",
            "REPORTDATE",
            "ADDRESS",
            "LATITUDE",
            "LONGITUDE"
        ],
        aliases=[
            "Crash ID:",
            "Report date:",
            "Address:",
            "Latitude:",
            "Longitude:"
        ],
        sticky=False
    )
).add_to(m_missing)

folium.LayerControl().add_to(m_missing)

m_missing

Zoom into the red points and investigate them.

Ask:

- Are they close to the edge of Washington, D.C.?
- Are they close to a census-tract boundary?
- Are they located on roads that run along a boundary?
- Does the point appear only slightly outside the census-tract polygons?

A failed `within` match does **not automatically mean that the crash record is wrong**. It can also arise from differences in how different spatial datasets represent boundaries.

The interactive map lets us inspect the records before deciding how to handle them.

## Save a Folium map

A Folium map is an HTML web map.

It can be saved using:

`map_name.save("filename.html")`

The HTML file can then be opened in a web browser.

In [ ]:
m.save(
    data_path + r"\DC_crashes_folium.html"
)

m_missing.save(
    data_path + r"\DC_unmatched_census_crashes.html"
)

print("Folium maps saved.")

# Class Exercise 16 — Map your waste-bin data using Folium

Return to the waste-bin point data collected earlier using QField.

Create an **interactive Folium map** of your own waste-bin observations.

### Tasks

1. Read your waste-bin spatial file using GeoPandas.
2. Inspect its CRS.
3. Convert it to `EPSG:4326` if necessary.
4. Create a Folium map centred on your study area.
5. Add the waste-bin points using `folium.GeoJson()`.
6. Use `CircleMarker` to display the points.
7. Create a tooltip containing **at least three useful fields** from your waste-bin dataset.
8. Add `LayerControl()`.
9. Save the final map as an HTML file.

### Things to think about

- Which attributes are most useful to show in the tooltip?
- Does the OpenStreetMap background provide enough context for interpreting the waste-bin locations?
- Do you need all observations on the map, or would filtering improve readability?
- How is the interactive Folium map different from the static maps you created earlier?

You do **not** need a `for` loop if your waste-bin locations are already stored as a GeoDataFrame. You can pass the complete point GeoDataFrame to `folium.GeoJson()`.

# Key ideas

- A spatial join can assign the same point observations to different polygon geographies.
- The original field names in GIS datasets often carry useful information and do not always need to be renamed.
- Ward, census-tract and neighborhood-cluster summaries can produce different spatial patterns.
- Raw counts and area-normalized densities answer different questions.
- Area must be calculated using an appropriate projected CRS.
- `zorder` controls which Matplotlib layer is drawn above another.
- Higher `zorder` values are drawn above lower values; only the relative ordering matters.
- Folium produces interactive web maps with zooming, panning, tooltips and layer controls.
- Folium uses OpenStreetMap as its default basemap when no tile layer is specified.
- `folium.GeoJson()` can add an entire GeoDataFrame at once, so a row-by-row `for` loop is not always necessary.
- Interactive maps are useful not only for presentation, but also for investigating unmatched or suspicious spatial records.